In [1]:
import h5py
from Utils import process_and_route_single

In [2]:
import numpy as np

In [3]:
summe_metadata = h5py.File('Data/Metadata/summe_metadata.h5')
tvsum_metadata = h5py.File('Data/Metadata/tvsum_metadata.h5')

In [4]:
user_score = summe_metadata['video_1']['user_score'][...]
user_summary = summe_metadata['video_1']['user_summary'][...]
positions = summe_metadata['video_1']['positions'][...]
shot_bounds = summe_metadata['video_1']['shot_bounds'][...]
n_frames = summe_metadata['video_1']['n_frames'][...]
pred = user_score[:,::15]

In [5]:
user_summary = user_summary[:,::15]

In [52]:
user_summary.shape

(15, 300)

In [37]:
pred.shape

(1, 300)

In [39]:
frame_init_scores = pred[0]
frame_scores = np.zeros(n_frames, dtype=np.float32)
if positions.dtype != int:
    positions = positions.astype(np.int32)

if positions[-1] != n_frames:
    positions = np.concatenate([positions, [n_frames]])

for i in range(len(positions) - 1):
    pos_left, pos_right = positions[i], positions[i + 1]
    if i == len(frame_init_scores):
        frame_scores[pos_left:pos_right] = 0
    else:
        frame_scores[pos_left:pos_right] = frame_init_scores[i]

In [40]:
from scipy.stats import kendalltau,spearmanr

In [42]:
kendalltau(frame_scores,user_score[0])

SignificanceResult(statistic=np.float64(0.9226601549033603), pvalue=np.float64(0.0))

In [29]:
positons

array([   0,   15,   30,   45,   60,   75,   90,  105,  120,  135,  150,
        165,  180,  195,  210,  225,  240,  255,  270,  285,  300,  315,
        330,  345,  360,  375,  390,  405,  420,  435,  450,  465,  480,
        495,  510,  525,  540,  555,  570,  585,  600,  615,  630,  645,
        660,  675,  690,  705,  720,  735,  750,  765,  780,  795,  810,
        825,  840,  855,  870,  885,  900,  915,  930,  945,  960,  975,
        990, 1005, 1020, 1035, 1050, 1065, 1080, 1095, 1110, 1125, 1140,
       1155, 1170, 1185, 1200, 1215, 1230, 1245, 1260, 1275, 1290, 1305,
       1320, 1335, 1350, 1365, 1380, 1395, 1410, 1425, 1440, 1455, 1470,
       1485, 1500, 1515, 1530, 1545, 1560, 1575, 1590, 1605, 1620, 1635,
       1650, 1665, 1680, 1695, 1710, 1725, 1740, 1755, 1770, 1785, 1800,
       1815, 1830, 1845, 1860, 1875, 1890, 1905, 1920, 1935, 1950, 1965,
       1980, 1995, 2010, 2025, 2040, 2055, 2070, 2085, 2100, 2115, 2130,
       2145, 2160, 2175, 2190, 2205, 2220, 2235, 22

In [11]:
metadata = {'positions':positions,'n_frames':n_frames,'shot_bounds':shot_bounds}
ground_truth_data = {'user_score':user_score,'user_summary':user_summary}

In [ ]:
ground_truth_data['user_summary'][0]

np.float32(652.0)

In [9]:
import torch

In [55]:
preds = torch.from_numpy(pred)

In [14]:
for i in range(len(user_score)):
    print(user_score[i].shape)

(4494,)


# Testing stuff 
- [ x ] GT
- [ x ] user_score
- [ x ] user_summary

done for all datasets 

In [56]:
from Utils import upsample,generate_summary_single,evaluate_f1,eval_kendall,eval_spearman
import numpy as np

In [59]:
def post_process_preds(pred,metadata,post_process):
    positions = metadata['positions']
    n_frames = metadata['n_frames']
    shot_bound = metadata['shot_bounds']
    if post_process =='none':
        return pred
    elif post_process == "upsample":
        return upsample(pred,positions,n_frames)
    elif post_process == "summary_gen":
        return generate_summary_single(shot_bound,pred,n_frames,positions)

def process_and_route_single(pred:torch.tensor,gt:torch.tensor,metadata:dict,ground_truth_data:dict,eval_type:str,post_process:str = "upsample",metric='corr'):
    pred = pred.to('cpu').squeeze().numpy()
    gt = gt.to('cpu').squeeze().numpy()
    pred = post_process_preds(pred,metadata,post_process) # Does the post-procesing based on type
    if metric =="corr":
        # Returns both Kendall and Spearman Correlation
        return evaluate_correlation(pred,gt,ground_truth_data,eval_type)
    if metric =="f1":
        post_process = "summary_gen"
        return evaluate_f1(pred,ground_truth_data,eval_type)


def evaluate_correlation(pred,gt,ground_truth_data,eval_type):
    if eval_type == 'gt':
        return {"kendall":eval_kendall(pred,gt),"spearman":eval_spearman(pred,gt)}
    if eval_type == "user_score":
        user_score = ground_truth_data['user_score']
        # We do list comprenehsion since tvsum has multiple user scores 
        all_kendalls = [eval_kendall(pred,gt_i) for gt_i in user_score]
        all_spearmans = [eval_spearman(pred,gt_i) for gt_i in user_score]

        return {"kendall":np.mean(all_kendalls),"spearman":np.mean(all_spearmans)}
    if eval_type == "user_summary":
        user_score = ground_truth_data['user_summary']
        print(user_score.shape)
        print(pred.shape)
        # We do list comprenehsion since all of them have multiple summaries
        all_kendalls = [eval_kendall(pred,gt_i) for gt_i in user_score]
        all_spearmans = [eval_spearman(pred,gt_i) for gt_i in user_score]

        return {"kendall":np.mean(all_kendalls),"spearman":np.mean(all_spearmans)}


In [60]:
process_and_route_single(preds,preds,metadata,ground_truth_data,eval_type='user_summary',post_process='none')

(15, 300)
(300,)


{'kendall': np.float64(0.40108225531168434),
 'spearman': np.float64(0.44935393226661674)}

## Testing F1 score eval


First testing F1 with max and average

In [6]:
eval_type = 'avg'
metric = 'f1'
post_process = 'binarize_top_k'

In [7]:
pred.shape[-1]

300

In [13]:
import numpy as np
from scipy.stats import spearmanr,kendalltau
from sklearn.metrics import f1_score
from Utils.post_process import knapSack, upsample
import h5py
import torch

In [18]:
def binarize_top_percent(score, top_percent=0.15):
    """
    Selects the top_percent frames of the video to be included in the summary, by binarizing the top percentage of scores into a binary indicator array.
    :param numpy.ndarray score: 1D array of scores for frames.
    :param float top_percent: Percentage of frames to be included in the summary. Defaults to 0.15.
    :return numpy.ndarray: Binary array with values 1.0 for the top_percent highest scores and 0.0 otherwise.
    """
    n = score.size
    k = int((top_percent * n))
    sorted_descending = np.sort(score)[::-1]
    threshold_val = sorted_descending[k]
    # Create binary array (values ≥ threshold become 1)
    binary_score = (score >= threshold_val).astype(np.float32)
    return binary_score
def eval_kendall(preds:np.ndarray,gt:np.ndarray):

    return kendalltau(preds,gt)[0]

def eval_spearman(preds:np.ndarray,gt:np.ndarray):

    return spearmanr(preds,gt)[0]


In [22]:

def post_process_preds(pred,metadata,post_process):
    positions = metadata['positions']
    n_frames = metadata['n_frames']
    shot_bound = metadata['shot_bounds']
    if post_process =='none':
        return pred
    elif post_process == "upsample":
        return upsample(pred,positions,n_frames)
    elif post_process == "summary_gen":
        return generate_summary_single(shot_bound,pred,n_frames,positions)
    elif post_process == "binarize_top_k":
        return binarize_top_percent(pred)
    else:
        raise ValueError('provide an eval type: [none,upsample,summary_gen]')

In [20]:
def process_and_route_single(pred:torch.tensor,gt:torch.tensor,metadata:dict,ground_truth_data:dict,eval_type:str,post_process:str = "upsample",metric='corr'):
    pred = pred.to('cpu').squeeze().numpy()
    gt = gt.to('cpu').squeeze().numpy()
    pred = post_process_preds(pred,metadata,post_process) # Does the post-procesing based on type
    if metric =="corr":
        # Returns both Kendall and Spearman Correlation
        return evaluate_correlation(pred,gt,ground_truth_data,eval_type)
    elif metric =="f1":
        assert post_process in ['summary_gen','binarize_top_k'], "For F1 evaluation Post processing must be summary_gen, or binarize_top_k"
        return evaluate_f1(pred,ground_truth_data,metadata,post_process,eval_type)
    else:
        raise ValueError('provide an eval type: [corr,f1]')
    

def evaluate_correlation(pred,gt,ground_truth_data,eval_type):
    if eval_type == 'gt':
        return {"kendall":eval_kendall(pred,gt),"spearman":eval_spearman(pred,gt)}
    if eval_type == "user_score":
        user_score = ground_truth_data['user_score']
        # We do list comprenehsion since tvsum has multiple user scores 
        all_kendalls = [eval_kendall(pred,gt_i) for gt_i in user_score]
        all_spearmans = [eval_spearman(pred,gt_i) for gt_i in user_score]

        return {"kendall":np.mean(all_kendalls),"spearman":np.mean(all_spearmans)}
    elif eval_type == "user_summary":
        user_score = ground_truth_data['user_summary']
        # We do list comprenehsion since all of them have multiple summaries
        all_kendalls = [eval_kendall(pred,gt_i) for gt_i in user_score]
        all_spearmans = [eval_spearman(pred,gt_i) for gt_i in user_score]

        return {"kendall":np.mean(all_kendalls),"spearman":np.mean(all_spearmans)}
    else:
        raise ValueError('provide an eval type: [gt,user_score,user_summary]')

    
#TODO: implement f1, which takes the eval_type to have the "max" and "best" eval thing from past research
def evaluate_f1(pred,ground_truth_data,metadata,post_process,eval_type):
    assert eval_type in ['max','avg'], "evaluation is done either max or average"
    user_summaries = ground_truth_data['user_summary']
    scores = []
    for user_summary in user_summaries:
        if post_process == 'binarize_top_k':
            print(user_summary.shape)
            if len(pred)!= len(user_summary):
                user_summary = user_summary[metadata['picks']]
            scores.append(f1_score(user_summary,pred))
        elif post_process == 'summary_gen':
            scores.append(f1_score(user_summary,pred))
        else:
            raise NotImplementedError
    if eval_type == "max":
        return np.max(scores)
    elif eval_type == "avg":
        return np.average(scores)
    else:
        raise NotImplementedError

In [23]:
process_and_route_single(torch.from_numpy(pred),torch.from_numpy(pred),metadata,ground_truth_data,eval_type,post_process,metric)

(300,)
(300,)
(300,)
(300,)
(300,)
(300,)
(300,)
(300,)
(300,)
(300,)
(300,)
(300,)
(300,)
(300,)
(300,)


np.float64(0.5393317952279639)

In [7]:
import json

with open('activity_net_to_video_id_map.json','r') as f:
    mapping_dict = json.load(f)

In [8]:
with open('Splits/videoxum_reduced_split.json') as f:
    split_dict = json.load(f)

In [9]:
split_dict = [{base_key: [f'video_xum/{mapping_dict[value]}' for value in split_dict[base_key]] for base_key in split_dict.keys()}]

KeyError: 'videoxum/v_QOlSCBRmfWY'

In [10]:
split_dict = [{base_key: [f'video_xum/{mapping_dict[value.split("/")[1]]}' for value in split_dict[base_key]] for base_key in split_dict.keys()}]

In [8]:
split_dict[0].keys()

dict_keys(['train', 'test', 'val'])

In [11]:
with open('Splits/script_videoxum_split_reduced_fixed.json','w') as f:
    json.dump(split_dict,f)

In [2]:
from Data.Datasets import MultiModalh5Loader

/home/aash/miniconda3/envs/MemorabilityEnv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
a = MultiModalh5Loader('Splits/script_videoxum_split_reduced_fixed.json','train',0,'blip',['frame_features','text_summary'])

In [4]:
import h5py

In [5]:
from Data.Loaders.collators import batch_text_sum_visual

In [6]:
from torch.utils.data import DataLoader

In [7]:
c = DataLoader(a,3,collate_fn=batch_text_sum_visual)

In [8]:
next(iter(c))

/home/aash/MLProjects/Seashore/Data/Loaders/collators.py:75: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return {"visual_features":torch.tensor(visual_features), "text_labels": textual_features


{'visual_features': tensor([[[-0.7559,  0.2810,  0.5762,  ...,  0.8447,  0.3962,  0.0523],
          [-0.7622,  0.2189,  0.5508,  ...,  0.9697,  0.3354,  0.1534],
          [-0.8989,  0.1694,  0.5571,  ...,  0.9849,  0.2189,  0.0502],
          ...,
          [-0.6621,  0.2705,  0.6724,  ...,  0.6348,  0.7432,  0.2927],
          [-0.6440,  0.2443,  0.5796,  ...,  0.4460,  0.8125,  0.3828],
          [-0.7876,  0.3589,  0.7456,  ...,  0.8179,  0.7915,  0.1903]],
 
         [[-1.6768,  0.5151,  0.0864,  ...,  0.3284,  0.7505,  0.3806],
          [-1.7266,  0.5723,  0.0388,  ...,  0.5220,  0.6328,  0.3188],
          [-1.7754,  0.6655, -0.0465,  ...,  0.5601,  0.7676, -0.0101],
          ...,
          [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
          [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
          [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000]],
 
         [[-0.3684,  0.3799,  0.0434,  ...,  0.5640,  0.5210,  0.1305],
         

In [34]:
def return_key_names(h5_files):
    with h5py.File(h5_files,'r') as f:
        print(f[list(f.keys())[0]].keys())

In [35]:
return_key_names('Data/h5Datasets/multi_modal/blip_video_xum.h5')

<KeysViewHDF5 ['activitynet_video_id', 'frame_features', 'gtscore', 'gtscore_xum', 'n_frames', 'positions', 'shot_bounds', 'text_summary', 'user_score', 'user_summary', 'vsum_time_stamps']>


In [38]:
next(iter(a))

{'gtscore': array([0. , 0. , 0. , 0. , 0. , 0. , 0.1, 0.1, 0.6, 0.5, 0.6, 0.6, 0.6,
        0.3, 0.2, 0.1, 0. , 0. , 0. , 0. , 0.1, 0.1, 0.1, 0.2, 0.1, 0.1,
        0.1, 0.1, 0.1, 0.1, 0.1, 0.2, 0.2, 0.2, 0.3, 0.3, 0.3, 0.2, 0.2,
        0.2, 0.2, 0.3, 0.2, 0.2, 0.2, 0.2, 0.1, 0. , 0. , 0. , 0. , 0. ,
        0. , 0. , 0. , 0. , 0. , 0.1, 0.2, 0.3, 0.4, 0.3, 0.3, 0.3, 0. ,
        0.1, 0.2, 0.2, 0.5, 0.5, 0.3, 0.4, 0.3, 0.1, 0.1, 0. , 0. , 0. ,
        0. , 0. , 0. , 0. , 0. ]),
 'data_point': 'video_xum/video_1',
 'frame_features': array([[-0.756  ,  0.281  ,  0.576  , ...,  0.8447 ,  0.3962 ,  0.05228],
        [-0.762  ,  0.2189 ,  0.551  , ...,  0.9697 ,  0.3354 ,  0.1534 ],
        [-0.899  ,  0.1694 ,  0.557  , ...,  0.985  ,  0.2189 ,  0.05017],
        ...,
        [-0.662  ,  0.2705 ,  0.6724 , ...,  0.635  ,  0.743  ,  0.2927 ],
        [-0.644  ,  0.2443 ,  0.5796 , ...,  0.446  ,  0.8125 ,  0.3828 ],
        [-0.7876 ,  0.359  ,  0.7456 , ...,  0.818  ,  0.7915 ,  0.1903 ]]

In [1]:
from Utils.h5_utils import *

In [10]:
h5_1 = 'Data/h5Datasets/siglip2/siglip2_tvsum.h5'
h5_2 = 'tvsum_siglip_text.h5'
h5_out = 'siglip2_multimodal_tvsum.h5'

In [12]:
import h5py

In [21]:
d = h5py.File('siglip2_multimodal_tvsum.h5')

In [23]:
d['video_1'].keys()

<KeysViewHDF5 ['features', 'gtscore', 'n_frames', 'positions', 'shot_bounds', 'text_features', 'user_score', 'user_summary']>

In [13]:
b = h5py.File(h5_2)

In [18]:
b['video_1'].keys()

<KeysViewHDF5 ['text_features']>

In [19]:
def combine_h5(h5_1_file,h5_2_file,final_h5):
    """
    Function that combines multiple h5 files 
    """
    with h5py.File(h5_1_file, "r") as h5_1, \
         h5py.File(h5_2_file, "r") as h5_2, \
         h5py.File(final_h5, "w") as h5_out:
        for key in h5_1.keys():
            h5_1.copy(key, h5_out)

        for key in h5_2.keys():
            print(key)
            for sub_key in h5_2[key].keys():
                h5_2[key].copy(sub_key, h5_out[key])

In [2]:
read_and_rename('siglip2_multimodal_tvsum.h5',{'features':'frame_features'})

In [20]:
combine_h5(h5_1,h5_2,h5_out)

video_1
video_10
video_11
video_12
video_13
video_14
video_15
video_16
video_17
video_18
video_19
video_2
video_20
video_21
video_22
video_23
video_24
video_25
video_3
video_4
video_5
video_6
video_7
video_8
video_9
